# 05 — ML Baseline and Advanced Evaluation

**DiscoveryRank Phase 3 — Integrating Advanced Metrics & SVD ML Baseline**

This notebook upgrades the evaluation framework to a production-like temporal split setting. We:
1. **Global Time Split**: Split the dataset into a strict contiguous `Train` (past) and `Test` (future) set to prevent data leakage.
2. **ML Baseline**: Train a Truncated SVD matrix factorization model (via scipy) on the `Train` set.
3. **Advanced Metrics**: Compute global popularity priors based *only* on the `Train` set to fuel advanced **Novelty** and **Serendipity** metrics.
4. **Explicit Candidate Sourcing**: Generate candidate pools for `Test` sessions, tracking exactly where each candidate came from (observed, popular, history, random).
5. **Holistic Evaluation**: Rank the candidate pools using the 3 baseline heuristics alongside the new `SVD` baseline, and evaluate them on Relevance, Freshness, Diversity, Repetition, Novelty, and Serendipity.

In [1]:
import sys
import os
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('../src'))
import ranking_strategies
import eval_metrics
import candidate_generation
import model_baselines

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Imports OK')

Imports OK


## 1. Global Temporal Split

In [2]:
sample_path = '../outputs/pipeline_sample.csv'
df = pd.read_csv(sample_path)
print(f'Loaded Master DF: {df.shape[0]:,} rows')

# Split chronologically to train the SVD model
# 80th percentile of time as the split point.
split_time = df['time_ms'].quantile(0.80)

train_df = df[df['time_ms'] <= split_time].copy()
test_df = df[df['time_ms'] > split_time].copy()

print(f"Train split: {len(train_df):,} interactions")
print(f"Test split: {len(test_df):,} interactions")

Loaded Master DF: 43,028 rows
Train split: 34,422 interactions
Test split: 8,606 interactions


## 2. Train SVD Baseline & Prepare Priors

In [3]:
# 2a. Train the SVD Matrix Factorization Model
try:
    svd_model = model_baselines.SVDBaseline(factors=50)
    svd_model.fit(train_df)
    ML_AVAILABLE = True
except Exception as e:
    print('SVD Training failed:', e)
    ML_AVAILABLE = False

# 2b. Compute strictly-historical popularity priors for Novelty evaluation
# Novelty formulation requires knowing P(item) = interactions(item) / total_interactions
item_popularity = train_df.groupby('video_id').size().to_dict()
total_train_interactions = len(train_df)

Fitting SVD model (k=50) on 993 users × 7338 items...
SVD model fit complete.


## 3. Generate Candidate Pools for Test Sessions

I instantiate the `SessionCandidateGenerator` strictly on the `train_df`. This ensures that when generating popular or historical candidates for the `test_df` sessions, it *cannot* peek into the future.

In [4]:
# The generator strictly uses Train data to construct candidate synthetics
generator = candidate_generation.SessionCandidateGenerator(train_df)

# Target a sample of test sessions 
# (Require users to have presence in Train so SVD isn't pure cold-start for everyone)
eligible_users = train_df['user_id'].unique()
test_sessions = test_df[test_df['user_id'].isin(eligible_users)][['user_id', 'session_id']].drop_duplicates()

# Sample down so evaluation runs reasonably fast
sampled_test_sessions = test_sessions.sample(min(500, len(test_sessions)), random_state=42)
print(f'Evaluating on {len(sampled_test_sessions)} test sessions.')

candidate_pools = {}
skipped = 0
POOL_SIZE = 100

for _, row in sampled_test_sessions.iterrows():
    uid, sid = row['user_id'], row['session_id']
    try:
        # Pull the ACTUAL session df from test_df to get the observed ground truths
        actual_session = test_df[(test_df['user_id'] == uid) & (test_df['session_id'] == sid)]
        
        # Temporarily inject this actual session into the generator so it can
        # attach the observed items and synthesize the rest from past history.
        generator.df = pd.concat([train_df, actual_session])
        
        pool = generator.generate_pool(uid, sid, pool_size=POOL_SIZE)
        if len(pool) >= 10:
            candidate_pools[(uid, sid)] = pool
        else:
            skipped += 1
    except Exception as e:
        skipped += 1

print(f'Successfully generated {len(candidate_pools)} pools (Skipped {skipped}).')
# Reset generator state
generator.df = train_df.copy()

Evaluating on 500 test sessions.


Successfully generated 500 pools (Skipped 0).


## 4. Run Ranking Strategies & Compute Advanced Metrics

I now evaluate `popularity`, `freshness`, `diversity`, and `svd_baseline` on the N=100 pools.

In [5]:
STRATEGY_FUNCS = {
    'popularity_based': ranking_strategies.popularity_based,
    'freshness_boosted': ranking_strategies.freshness_boosted,
    'diversity_aware_rerank': ranking_strategies.diversity_aware_rerank,
}

if ML_AVAILABLE:
    # lambda wrapper to inject the fitted model
    STRATEGY_FUNCS['svd_baseline'] = lambda p: model_baselines.svd_ranker(p, svd_model)

all_results = []
top_k_coverage_sets = {s: set() for s in STRATEGY_FUNCS.keys()}

for (uid, sid), pool_df in candidate_pools.items():
    for strategy_name, strategy_fn in STRATEGY_FUNCS.items():
        # Rank pool
        ranked = strategy_fn(pool_df)
        
        # Reattach required feature columns
        feature_cols = ['video_id', 'y_relevant', 'implicit_completion_ratio', 'item_age_days', 'author_id', 'tag', 'is_observed_in_session', 'candidate_source']
        available_features = [c for c in feature_cols if c in pool_df.columns]
        # Drop to prevent suffix collision
        drop_cols = [c for c in available_features if c in ranked.columns and c != 'video_id']
        ranked = ranked.drop(columns=drop_cols, errors='ignore')
        ranked_full = ranked.merge(pool_df[available_features], on='video_id', how='left')
        
        # Calculate metrics including Novelty/Serendipity
        metrics = eval_metrics.score_all_metrics(
            ranked_full, 
            strategy_name=strategy_name, 
            k=20,  
            item_popularity_dict=item_popularity,
            total_train_interactions=total_train_interactions
        )
        
        # Track global item coverage for Top 20
        top_20_items = ranked_full.sort_values('rank').head(20)['video_id'].dropna().tolist()
        top_k_coverage_sets[strategy_name].update(top_20_items)
        
        # Source Tracking: which sources made it into the Top 20?
        if 'candidate_source' in ranked_full.columns:
            top_20_sources = ranked_full.sort_values('rank').head(20)['candidate_source'].value_counts(normalize=True).to_dict()
            metrics['source_observed_pct'] = top_20_sources.get('observed', 0.0)
            metrics['source_popular_pct'] = top_20_sources.get('popular', 0.0)
            metrics['source_history_pct'] = top_20_sources.get('history', 0.0)
            metrics['source_random_pct'] = top_20_sources.get('random', 0.0)
        
        metrics['user_id'] = uid
        metrics['session_id'] = sid
        all_results.append(metrics)

results_df = pd.DataFrame(all_results)
print(f'Evaluation complete: {len(results_df)} rows logged.')

Evaluation complete: 2000 rows logged.


## 5. Global Comparison and Tradeoff Analysis

In [6]:
# Calculate final Global Catalog Coverage
total_train_catalog = len(item_popularity)
coverage_metrics = []
for s_name, unique_items in top_k_coverage_sets.items():
    cov = len(unique_items) / total_train_catalog if total_train_catalog > 0 else 0
    coverage_metrics.append({'strategy': s_name, 'global_coverage_ratio': cov})
cov_df = pd.DataFrame(coverage_metrics)

# Aggregate session metrics
metric_cols = [
    'rel_mean_y_relevant',
    'fresh_mean_item_age_days',
    'div_unique_author_ratio',
    'div_unique_tag_ratio',
    'rep_consecutive_tag_rate',
    'adv_mean_novelty',
    'adv_mean_serendipity',
    'source_observed_pct',
    'source_popular_pct',
    'source_history_pct'
]
available_cols = [c for c in metric_cols if c in results_df.columns]

comparison_table = results_df.groupby('strategy')[available_cols].mean().reset_index()

# Merge Coverage
comparison_table = comparison_table.merge(cov_df, on='strategy', how='left')

strategy_order = ['popularity_based', 'freshness_boosted', 'diversity_aware_rerank', 'svd_baseline']
comparison_table['sort_idx'] = comparison_table['strategy'].map({s: i for i, s in enumerate(strategy_order)})
comparison_table = comparison_table.sort_values('sort_idx').drop(columns='sort_idx').set_index('strategy')

print('=== Phase 3 Strategy Comparison @K=20 ===')
display(comparison_table.round(4))

# Export
comparison_table.to_csv('../outputs/phase3_strategy_comparison.csv')
results_df.to_csv('../outputs/phase3_results_per_session.csv', index=False)
print('Outputs saved.')

=== Phase 3 Strategy Comparison @K=20 ===


,rel_mean_y_relevant,fresh_mean_item_age_days,div_unique_author_ratio,div_unique_tag_ratio,rep_consecutive_tag_rate,adv_mean_novelty,adv_mean_serendipity,source_observed_pct,source_popular_pct,source_history_pct,global_coverage_ratio
strategy,,,,,,,,,,,
popularity_based,0.0070,27.9210,0.9840,0.3508,0.0434,12.3775,0.0182,0.1244,0.4089,0.0372,0.3419
freshness_boosted,0.0070,27.9210,0.9840,0.3508,0.0466,12.3775,0.0182,0.1206,0.4212,0.0470,0.3434
diversity_aware_rerank,0.0070,27.9521,0.9941,0.7917,0.0086,12.4522,0.0912,0.1244,0.4089,0.0372,0.3419
svd_baseline,0.0020,27.9210,0.9840,0.3508,0.0465,12.3775,0.0182,0.0145,0.5746,0.1486,0.3023


Outputs saved.


## 6. Analysis on SVD ML Baseline

On sparse, highly skewed short-video data, implicit matrix factorization may or may not beat a strong popularity baseline in raw relevance. The goal is to observe *where* the model diverges in tradeoffs.

Look closely at **Novelty** and **Global Coverage**: Learned factor models typically surface items tailored to user-specific niches (history-adjacent sources), potentially increasing structural novelty and catalog coverage compared to pure popularity logic.